In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_INVESTMENT_INDEX")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_investment_links.parquet", index=False)

In [5]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 100


In [6]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\(\s*(\[\[.*?\]\])\s*\);'

sleep_duration = 5

In [7]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [8]:
# DataFrames de salida
dfs_quarters = []
dfs_months = []

start_time = time.time()
for url in country_links:
    country = urlparse(url).path.strip("/").split("/")[0]

    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            # --- separar según formato de Period ---
            if df["Period"].str.contains(r"Q\d").any():
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_quarters.append(df_pivot)
                print(f"✅ Datos trimestrales extraídos para {country}")
            else:
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_months.append(df_pivot)
                print(f"✅ Datos mensuales extraídos para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

# unir cada grupo por separado
df_quarters = pd.concat(dfs_quarters, axis=1) if dfs_quarters else pd.DataFrame()
df_months = pd.concat(dfs_months, axis=1) if dfs_months else pd.DataFrame()

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
✅ Datos trimestrales extraídos para Australia
✅ Datos trimestrales extraídos para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos trimestrales extraídos para Belgium
⏭️ País Bermuda no está en la lista, se omite.
⏭️ País Bolivia no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
⏭️ País Botswana no está en la lista, se omite.
✅ Datos trimestrales extraídos para Brazil
⏭️ País Brunei no está en la lista, se omite.
✅ Datos trimestrales extraídos para Bulgaria
⏭️ País Cameroon no está en la lista, se omite.
✅ Datos trimestrales extraídos para Canada
⏭️ País Cape-Verde no está en la lista, se omite.
✅ Datos trimestrales extraídos para Chile
✅ Datos tri

In [11]:
# resultados finales
print("\n📊 DataFrame trimestral")
df_quarters.tail()


📊 DataFrame trimestral


Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,Croatia,Cyprus,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Q4 2021,137.21,28.62,34.14,415.75,7.27,155.82,16663.77,40851.39,3.13,1.03,...,2.87,216.41,191300.0,67.56,394.04,52.53,974.94,673.67,1064.80,105.52
Q4 2022,150.88,31.87,39.07,471.68,9.60,163.34,18622.28,53464.40,3.77,1.29,...,3.22,252.40,203515.2,72.14,427.02,53.49,1060.31,1413.06,1154.60,115.28
Q4 2023,166.24,31.96,39.58,456.88,11.62,168.28,18024.71,37473.07,4.41,1.31,...,3.38,270.37,201998.3,77.38,450.79,53.42,1040.69,2723.62,1261.53,120.04
Q4 2024,175.44,32.51,42.58,528.20,12.70,179.27,20073.88,54587.21,5.04,2.17,...,3.27,270.71,203497.9,84.12,468.02,53.99,1091.88,3972.42,1318.78,125.68
Q2 2025,NaN,NaN,NaN,NaN,NaN,NaN,19336.88,75717.18,NaN,NaN,...,3.47,NaN,NaN,83.89,NaN,NaN,985.77,NaN,1350.68,NaN


In [10]:
print("\n📊 DataFrame mensual")
df_months.head()


📊 DataFrame mensual


""


In [12]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [13]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [14]:
df_paraguay.tail()

Country,Paraguay
Period,
Q4 2020,14142.68
Q4 2021,17224.76
Q4 2022,15838.54
Q4 2023,16259.69
Q4 2024,18220.38


In [15]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
new_quarter_df = pd.merge(df_quarters, df_paraguay, how="outer", left_index=True, right_index=True)
new_quarter_df.head()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,Croatia,Cyprus,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Q1 1960,1.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,5.84,NaN,NaN,NaN,NaN,NaN,21.33,1.07,NaN
Q1 1961,1.23,NaN,NaN,NaN,NaN,1.89,NaN,NaN,NaN,NaN,...,NaN,7.20,NaN,NaN,NaN,NaN,NaN,20.23,1.24,NaN
Q1 1962,1.24,NaN,NaN,NaN,NaN,2.01,NaN,NaN,NaN,NaN,...,NaN,9.04,NaN,NaN,NaN,NaN,NaN,22.18,1.28,NaN
Q1 1963,1.34,NaN,NaN,NaN,NaN,2.11,NaN,NaN,NaN,NaN,...,NaN,14.79,NaN,NaN,NaN,NaN,NaN,23.20,1.24,NaN
Q1 1964,1.46,NaN,NaN,NaN,NaN,2.54,NaN,NaN,NaN,NaN,...,NaN,18.57,NaN,NaN,NaN,NaN,NaN,26.33,1.60,NaN


In [16]:
def split_period_index(df):
    """
    Recibe un DataFrame con índice 'Period' (ej: 'Apr 1960' o 'Q1 1990')
    Devuelve el mismo DataFrame con columnas 'Year' y 'Month' o 'Quarter'
    """
    df = df.copy().reset_index()  # sacar Period del índice
    df["Year"] = None
    df["Month"] = None
    df["Quarter"] = None

    for i, period in enumerate(df["Period"]):
        parts = period.split()

        if len(parts) == 2:
            first, year = parts
            df.at[i, "Year"] = int(year)

            if "Q" in first:  # Formato trimestral
                df.at[i, "Quarter"] = first
            else:  # Formato mensual
                df.at[i, "Month"] = first

    return df.set_index("Period")  # opcional: devolver con Period otra vez de índice

In [17]:
def quarter_to_int(df):
    """
    Convierte los valores de la columna 'Quarter' a enteros (Q1 → 1, Q2 → 2, etc.)
    """
    df = df.copy()

    q_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}

    df["Quarter"] = df["Quarter"].map(q_map).fillna(df["Quarter"])

    return df

In [18]:
def year_to_int(df):
    """
    Convierte la columna 'Year' a enteros.
    Si hay valores no numéricos, los convierte en NaN.
    """
    df = df.copy()
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

In [20]:
final_df = new_quarter_df.copy()
final_df = split_period_index(final_df)
final_df = quarter_to_int(final_df)
final_df = year_to_int(final_df)

In [21]:
final_df.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,Croatia,Cyprus,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Q4 2020,119.24,26.87,33.03,377.41,7.64,138.00,12652.95,31244.21,3.02,1.58,...,351.05,49.97,951.16,438.67,966.48,96.80,14142.68,2020,None,4
Q4 2021,137.21,28.62,34.14,415.75,7.27,155.82,16663.77,40851.39,3.13,1.03,...,394.04,52.53,974.94,673.67,1064.80,105.52,17224.76,2021,None,4
Q4 2022,150.88,31.87,39.07,471.68,9.60,163.34,18622.28,53464.40,3.77,1.29,...,427.02,53.49,1060.31,1413.06,1154.60,115.28,15838.54,2022,None,4
Q4 2023,166.24,31.96,39.58,456.88,11.62,168.28,18024.71,37473.07,4.41,1.31,...,450.79,53.42,1040.69,2723.62,1261.53,120.04,16259.69,2023,None,4
Q4 2024,175.44,32.51,42.58,528.20,12.70,179.27,20073.88,54587.21,5.04,2.17,...,468.02,53.99,1091.88,3972.42,1318.78,125.68,18220.38,2024,None,4


In [22]:
final_df.dtypes

Country
Australia         float64
Austria           float64
Belgium           float64
Brazil            float64
Bulgaria          float64
Canada            float64
Chile             float64
Colombia          float64
Croatia           float64
Cyprus            float64
Czech-Republic    float64
Denmark           float64
Estonia           float64
Finland           float64
France            float64
Germany           float64
Greece            float64
Hong-Kong         float64
Hungary           float64
Iceland           float64
India             float64
Indonesia         float64
Ireland           float64
Israel            float64
Italy             float64
Japan             float64
Latvia            float64
Lithuania         float64
Luxembourg        float64
Malaysia          float64
Malta             float64
Mexico            float64
Netherlands       float64
New-Zealand       float64
Norway            float64
Peru              float64
Philippines       float64
Poland            float64
Port

In [23]:
final_df.to_parquet("../../data/quarter_investment_index_values.parquet",index=False)